In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/04.Gold_Helper

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
races_table    = f"{catalog_name}.{silver_schema}.races"
circuits_table = f"{catalog_name}.{silver_schema}.circuits"

target_table = f"{catalog_name}.{gold_schema}.dim_races"

In [0]:
races_df = spark.read.table(races_table).filter(col("batch_id")==v_batch_id)
circuits_df = spark.read.table(circuits_table).filter(col("batch_id")==v_batch_id)


In [0]:
from pyspark.sql.functions import *

In [0]:
races_circuits_df=(

                    races_df
                        .join(circuits_df, races_df.circuit_id == circuits_df.circuit_id, "inner")
                        .select(
                            races_df.season,
                            races_df.round,
                            races_df.race_name,
                            races_df.race_date,
                            circuits_df.circuit_name,
                            circuits_df.city,
                            circuits_df.country
                        )

                    )

In [0]:
columns_to_update=[
        "race_name",
        "race_date",
        "circuit_name",
        "city",
        "country"
    ]

In [0]:
write_to_gold(input_df=races_circuits_df,target_table=target_table,
                merge_condn="t.season = s.season AND t.round = s.round",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.table(target_table))